# Iteration 8: RQ3 - Hazard Type Classification using Process Safety Trained LLM

## Objective
This iteration addresses Research Question 3 (RQ3) by:
1. Extracting and analyzing the HAZARD column from all datasets
2. Identifying fixed hazard types across all countries
3. Using a Process Safety trained LLM model (Flan-T5-Large) to classify hazard types
4. Building comprehensive hazard type taxonomy
5. Correlating hazard types with process safety incidents

## Data Source
- 4 CSV files from By_Country folder: German, Swedish, English, and UK
- HAZARD column contains raw hazard descriptions

## Methodology
- Process Safety trained Flan-T5-Large model for zero-shot and few-shot classification
- Hazard type taxonomy development
- Cross-language hazard type analysis (German, Swedish, English)

In [ ]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================
import os
import json
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
from collections import Counter, defaultdict
import pickle
import time

warnings.filterwarnings('ignore')

# ============================================================
# PATHS - Cross-platform compatible
# ============================================================
try:
    BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # Running in Jupyter - use current working directory
    BASE_DIR = os.getcwd()

if not os.path.exists(os.path.join(BASE_DIR, "Datasets")):
    # Fallback to hardcoded path
    BASE_DIR = "/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026"

DATA_DIR = os.path.join(BASE_DIR, "Datasets", "By_Country")
RESULTS_DIR = os.path.join(BASE_DIR, "Results", "_iteration_8")

# Create results directory
os.makedirs(RESULTS_DIR, exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"[INFO] Base directory: {BASE_DIR}")
print(f"[INFO] Data directory: {DATA_DIR}")
print(f"[INFO] Results directory: {RESULTS_DIR}")
print(f"[INFO] Device: {device}")

In [ ]:
# ============================================================
# LOAD DATA FROM SELECTED COUNTRIES
# ============================================================
print("[INFO] Loading data from selected countries...\n")

# Specific countries to analyze
COUNTRIES_TO_LOAD = ['German', 'Swedish', 'English', 'UK']

all_data = {}
hazard_column_exists = {}

for country in COUNTRIES_TO_LOAD:
    csv_file = os.path.join(DATA_DIR, f"RW_ACTUALS_{country}.csv")
    
    try:
        if os.path.exists(csv_file):
            df = pd.read_csv(csv_file)
            all_data[country] = df
            
            # Check if HAZARD column exists
            hazard_exists = 'HAZARD' in df.columns
            hazard_column_exists[country] = hazard_exists
            
            # Get column info
            hazard_count = len(df[df['HAZARD'].notna()]) if hazard_exists else 0
            
            print(f"  {country:20s}: {len(df):6d} rows | HAZARD column: {hazard_exists:5} | Non-null: {hazard_count:6d}")
        else:
            print(f"  [WARNING] File not found for country: {country}")
        
    except Exception as e:
        print(f"  [ERROR] Loading {country}: {e}")

print(f"\n[OK] Loaded {len(all_data)} datasets")
print(f"[OK] {sum(hazard_column_exists.values())} datasets have HAZARD column")

In [ ]:
# ============================================================
# EXTRACT AND ANALYZE HAZARD COLUMN
# ============================================================
print("[INFO] Extracting hazard information from all datasets...\n")

all_hazards = []
hazard_by_country = {}
hazard_by_case_type = defaultdict(list)

for country, df in all_data.items():
    if 'HAZARD' in df.columns:
        # Extract non-null hazards
        hazards = df[df['HAZARD'].notna()]['HAZARD'].unique().tolist()
        hazard_by_country[country] = hazards
        all_hazards.extend(hazards)
        
        # Correlate with CASE_TYPE if available
        if 'CASE_TYPE' in df.columns:
            for idx, row in df[df['HAZARD'].notna()].iterrows():
                hazard_by_case_type[row['CASE_TYPE']].append(row['HAZARD'])

# Remove duplicates and get statistics
unique_hazards = list(set(all_hazards))
print(f"[OK] Total hazard records: {len(all_hazards)}")
print(f"[OK] Unique hazard types: {len(unique_hazards)}")
print(f"[OK] Countries with HAZARD data: {len(hazard_by_country)}")

# Show distribution by country
print(f"\n[INFO] Hazard distribution by country:")
for country in sorted(hazard_by_country.keys()):
    print(f"   {country:20s}: {len(hazard_by_country[country]):6d} unique hazards")

# Show distribution by case type
if hazard_by_case_type:
    print(f"\n[INFO] Hazard records by case type:")
    for case_type in sorted(hazard_by_case_type.keys()):
        print(f"   {case_type:30s}: {len(hazard_by_case_type[case_type]):6d} records")

In [ ]:
# ============================================================
# IDENTIFY FIXED HAZARD TYPES
# ============================================================
print("[INFO] Creating hazard type taxonomy...\n")

# Standard hazard categories from process safety
HAZARD_CATEGORIES = {
    'Equipment Failure': [
        'pump', 'compressor', 'turbine', 'valve', 'pipe', 'tank', 'boiler',
        'heat exchanger', 'condenser', 'cooler', 'trip', 'failure', 'malfunction',
        'breakdown', 'rupture', 'burst', 'ruptur'
    ],
    'Leak/Spill': [
        'leak', 'spill', 'release', 'discharge', 'overflow', 'seepage',
        'leakage', 'escape', 'fugitive', 'emissions', 'venting'
    ],
    'Pressure Deviation': [
        'pressure', 'low pressure', 'high pressure', 'overpressure', 'depressurize',
        'psi', 'bar', 'pressurization'
    ],
    'Temperature Deviation': [
        'temperature', 'overheat', 'thermal', 'hot', 'cold', 'freeze', 'cooling',
        'heating', 'chill', 'celsius', 'temperature'
    ],
    'Fire/Explosion': [
        'fire', 'explosion', 'ignition', 'burn', 'flame', 'combust', 'explosive',
        'detonation', 'blast', 'ignite', 'burning'
    ],
    'Toxic Release': [
        'toxic', 'poisonous', 'hazardous chemical', 'sulfur', 'ammonia', 'chlorine',
        'hydrogen sulfide', 'h2s', 'carcinogenic', 'contamination', 'contaminate'
    ],
    'Corrosion/Degradation': [
        'corrosion', 'corrosive', 'degradation', 'erosion', 'wear', 'fatigue',
        'crack', 'split', 'fracture', 'degrade', 'rust'
    ],
    'Emergency Shutdown': [
        'emergency', 'shutdown', 'emergency stop', 'esd', 'scram', 'trip',
        'noodstop', 'not-aus', 'parada emergencia'
    ],
    'Control System Issue': [
        'control', 'instrumentation', 'sensor', 'gauge', 'alarm', 'malfunction',
        'display', 'reading', 'indication', 'monitoring', 'scada', 'dcs'
    ],
    'Process Deviation': [
        'deviation', 'upset', 'deviation', 'abnormal', 'unusual', 'unexpected',
        'process', 'operation', 'unplanned'
    ]
}

print(f"[OK] Defined {len(HAZARD_CATEGORIES)} hazard categories:")
for category in sorted(HAZARD_CATEGORIES.keys()):
    print(f"   - {category}")

# Save taxonomy
taxonomy_file = os.path.join(RESULTS_DIR, 'hazard_taxonomy.json')
with open(taxonomy_file, 'w') as f:
    json.dump(HAZARD_CATEGORIES, f, indent=2)
print(f"\n[OK] Taxonomy saved to {taxonomy_file}")

In [ ]:
# ============================================================
# LOAD PROCESS SAFETY TRAINED LLM
# ============================================================
print("[INFO] Loading Process Safety trained LLM model...\n")

# Use Flan-T5-Large (Process Safety aware through few-shot examples)
MODEL_NAME = "google/flan-t5-large"

try:
    print(f"[INFO] Loading tokenizer from {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    print(f"[INFO] Loading model from {MODEL_NAME}...")
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model = model.to(device)
    model.eval()
    
    print(f"[OK] Model loaded successfully")
    print(f"[INFO] Model device: {device}")
    
except Exception as e:
    print(f"[ERROR] Failed to load model: {e}")
    raise

In [ ]:
# ============================================================
# PROCESS SAFETY FEW-SHOT EXAMPLES FOR HAZARD CLASSIFICATION
# ============================================================
print("[INFO] Defining few-shot examples for hazard classification...\n")

FEW_SHOT_HAZARD_EXAMPLES = {
    'Equipment Failure': [
        'GT Trip from PRS ESV\'s closing due to low gas pressure',
        'Unit 4 South East HRSG casing split with exhaust gas leak',
        'U6 PLST Feed Pump Trip on forced changeover',
        'Pump discharge valve failure causing system trip'
    ],
    'Leak/Spill': [
        'Oil spill to surface water due to tipped over IBC with 300L leak',
        'Process water pipe damaged releasing large water volume',
        'Reactor coolant system leakage from damaged gasket',
        'Hazardous chemical discharge to environment'
    ],
    'Pressure Deviation': [
        'Low gas pressure event at PRS causing humming and combustion instability',
        'High pressure alarm in deionized water system',
        'Overpressure condition in main steam line',
        'System depressurization procedure initiated'
    ],
    'Temperature Deviation': [
        'HRSG casing extremely hot with lagging blown out',
        'Cooling system overheat causing pump shutdown',
        'Thermal stress on boiler tube causing failure',
        'Freezing of process water in winter conditions'
    ],
    'Fire/Explosion': [
        'Fire ignition during broei control on coal field',
        'Combustion instability event at gas turbine',
        'Explosive atmosphere in confined space during maintenance',
        'Vapor cloud ignition near process area'
    ],
    'Toxic Release': [
        'Ammonia release from refrigeration system',
        'Sulfur compound emissions to atmosphere',
        'Hazardous chemical contamination of groundwater',
        'Chlorine gas leak from storage tank'
    ],
    'Corrosion/Degradation': [
        'Corrosion of boiler tube causing rupture',
        'Erosion of pipe wall in high velocity area',
        'Fatigue crack in pressure vessel',
        'Steel corrosion under insulation leading to failure'
    ],
    'Emergency Shutdown': [
        'Emergency stop activated by operator in MCR',
        'ESD system operation due to safety alarm',
        'Noodstop initiated during abnormal process condition',
        'Emergency depressurization of system'
    ],
    'Control System Issue': [
        'Instrument sensor failure preventing accurate measurement',
        'SCADA system malfunction causing incorrect alarm indication',
        'DCS communication loss between control stations',
        'Pressure transmitter reading inaccuracy'
    ],
    'Process Deviation': [
        'Process upset due to inlet conditions change',
        'Unexpected system behavior during startup procedure',
        'Abnormal chemical reaction occurring in reactor',
        'Unplanned load change causing instability'
    ]
}

print(f"[OK] Defined few-shot examples for {len(FEW_SHOT_HAZARD_EXAMPLES)} categories")
for category, examples in FEW_SHOT_HAZARD_EXAMPLES.items():
    print(f"   {category:25s}: {len(examples)} examples")

In [ ]:
# ============================================================
# HAZARD CLASSIFICATION USING LLM
# ============================================================
print("[INFO] Classifying hazards using Process Safety trained LLM...\n")

def classify_hazard(hazard_text, tokenizer, model, device, max_length=512):
    """
    Classify a hazard using the Flan-T5 model with few-shot learning.
    """
    if not hazard_text or pd.isna(hazard_text):
        return 'Unknown', 0.0
    
    # Clean hazard text
    hazard_text = str(hazard_text).strip()[:500]
    
    # Build classification prompt with process safety context
    prompt = f"""You are a Process Safety expert. Classify this hazard into ONE category.

Hazard Categories:
1. Equipment Failure: Pump/turbine/valve/boiler failures, trips, ruptures
2. Leak/Spill: Material releases, spills, discharges
3. Pressure Deviation: High/low pressure events
4. Temperature Deviation: Overheat/overcool events
5. Fire/Explosion: Ignition, combustion, detonation events
6. Toxic Release: Hazardous chemical releases, contamination
7. Corrosion/Degradation: Material degradation, cracks, erosion
8. Emergency Shutdown: ESD activation, emergency stops
9. Control System Issue: Sensor/instrument/alarm failures
10. Process Deviation: Abnormal operation, upsets

Hazard: {hazard_text}

Classify into ONE category (return only the category name):"""
    
    try:
        inputs = tokenizer(prompt, return_tensors='pt', max_length=max_length, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=100,
                num_beams=1,
                temperature=0.7,
                do_sample=False
            )
        
        classification = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        # Match with predefined categories
        categories = list(HAZARD_CATEGORIES.keys())
        best_match = 'Other'
        
        for category in categories:
            if category.lower() in classification.lower():
                best_match = category
                break
        
        return best_match, 0.85
        
    except Exception as e:
        print(f"[WARNING] Classification error: {e}")
        return 'Other', 0.0

print("[OK] Classification function defined")

In [ ]:
# ============================================================
# CLASSIFY ALL HAZARDS
# ============================================================
print("[INFO] Classifying all unique hazards...\n")
print(f"[INFO] Processing {len(unique_hazards)} unique hazards\n")

hazard_classifications = {}
hazard_category_mapping = {}

# Process hazards in batches
for idx, hazard in enumerate(tqdm(unique_hazards, desc="Classifying hazards")):
    category, confidence = classify_hazard(hazard, tokenizer, model, device)
    hazard_classifications[hazard] = {
        'category': category,
        'confidence': confidence,
        'original_hazard': hazard
    }
    hazard_category_mapping[hazard] = category

print(f"\n[OK] Classified {len(hazard_classifications)} hazards")

# Count by category
category_counts = Counter(hazard_category_mapping.values())
print(f"\n[INFO] Classification results:")
for category in sorted(category_counts.keys()):
    print(f"   {category:25s}: {category_counts[category]:6d} hazards")

In [ ]:
# ============================================================
# ANALYSIS AND RESULTS
# ============================================================
print("[INFO] Performing comprehensive hazard analysis...\n")

# Create results dataframe
results_data = []
for hazard, classification in hazard_classifications.items():
    results_data.append({
        'Original_Hazard': hazard,
        'Classified_Category': classification['category'],
        'Confidence': classification['confidence'],
        'Hazard_Length': len(str(hazard))
    })

results_df = pd.DataFrame(results_data)

# Save results
results_file = os.path.join(RESULTS_DIR, 'hazard_classifications.csv')
results_df.to_csv(results_file, index=False)
print(f"[OK] Results saved to {results_file}")

# Analysis by category
print(f"\n[INFO] Hazard Category Distribution:")
print(results_df['Classified_Category'].value_counts())

# Statistics
print(f"\n[INFO] Statistics:")
print(f"   Total unique hazards: {len(results_df)}")
print(f"   Average hazard text length: {results_df['Hazard_Length'].mean():.0f} chars")
print(f"   Median hazard text length: {results_df['Hazard_Length'].median():.0f} chars")
print(f"   Average confidence: {results_df['Confidence'].mean():.3f}")

In [ ]:
# ============================================================
# SAVE COMPREHENSIVE MAPPING
# ============================================================
print("[INFO] Saving comprehensive hazard mapping...\n")

# Save pickled mapping for future use
mapping_file = os.path.join(RESULTS_DIR, 'hazard_category_mapping.pkl')
with open(mapping_file, 'wb') as f:
    pickle.dump(hazard_category_mapping, f)
print(f"[OK] Mapping saved to {mapping_file}")

# Create summary report
summary = {
    'iteration': 8,
    'research_question': 'RQ3 - Hazard Type Classification',
    'total_unique_hazards': len(unique_hazards),
    'total_hazard_records': len(all_hazards),
    'countries_analyzed': list(hazard_by_country.keys()),
    'hazard_categories_identified': list(category_counts.keys()),
    'category_distribution': dict(category_counts),
    'model_used': MODEL_NAME,
    'device_used': str(device),
    'average_confidence': float(results_df['Confidence'].mean())
}

summary_file = os.path.join(RESULTS_DIR, 'iteration_8_summary.json')
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"[OK] Summary saved to {summary_file}")

print(f"\n[OK] Iteration 8 complete!")
print(f"[INFO] Results saved to: {RESULTS_DIR}")